# Entrevista completa de Malvinas — entrega privada en Drive

Este notebook procesa los **tres originales completos**, sin publicarlos ni modificar sus archivos. No utiliza APIs pagas. Se ejecuta dentro de **tu cuenta de Google Colab** y guarda MP4/SRT en una carpeta nueva privada de Mi unidad.

1. Abrí una copia en Colab, conectá un entorno y ejecutá las celdas en orden.
2. Autorizá Drive con la cuenta que puede abrir la carpeta fuente.
3. Mirá los principios/finales de los tres videos y confirmá su orden cuando se solicite; la numeración no se toma como prueba.
4. Esperá la transcripción y exportación. Revisá las muestras con audio y subtítulos antes de confirmar la entrega.

El video completo **todavía no se produjo en Codex**: allí Drive requiere iniciar sesión y redirige a un dominio bloqueado. Este notebook es la alternativa ejecutable. Las pruebas locales del motor usan videos sintéticos y ASR simulado; no sustituyen la revisión del testimonio real.

Los resultados son `ENTREVISTA_MALVINAS_FINAL.mp4` y `ENTREVISTA_MALVINAS_SUBTITULOS.srt`. El reconocimiento automático puede equivocarse en nombres, fechas o lugares: se guardará una lista de revisión y el SRT editable, sin inventar correcciones.


In [ ]:
# 1 — Dependencias mínimas de esta sesión Colab
import sys, subprocess, os, shutil
assert 'google.colab' in sys.modules or os.path.isdir('/content'), 'Abrí este archivo en Google Colab.'
subprocess.run([sys.executable,'-m','pip','install','-q','faster-whisper==1.2.1','pysubs2==1.8.0','google-api-python-client>=2,<3'],check=True)
if not shutil.which('ffmpeg'):
    subprocess.run(['apt-get','update','-qq'],check=True)
    subprocess.run(['apt-get','install','-y','ffmpeg','fonts-dejavu-core'],check=True)
filters=subprocess.run(['ffmpeg','-hide_banner','-filters'],capture_output=True,text=True,check=True).stdout
assert ' ass ' in filters and 'loudnorm' in filters, 'FFmpeg necesita libass y loudnorm.'
os.environ['HF_HOME']='/content/video_studio_hf'
os.environ['HF_XET_CACHE']='/content/video_studio_hf/xet'
print('FFmpeg/libass y dependencias disponibles. No se utiliza ninguna API paga.')


In [ ]:
"""Self-contained media engine embedded in the private Colab notebook.
No credentials, original media, transcripts or API calls are embedded here.
"""
from pathlib import Path
from fractions import Fraction
import hashlib
import json
import math
import os
import re
import shutil
import subprocess
import tempfile
import textwrap


def run(args):
    p = subprocess.run([str(x) for x in args], capture_output=True, text=True)
    if p.returncode:
        raise RuntimeError(p.stderr[-5000:])
    return p.stdout


def probe(path):
    return json.loads(run(['ffprobe', '-v', 'error', '-show_format', '-show_streams', '-of', 'json', path]))


def checksum(path, algorithm='sha256'):
    h = hashlib.new(algorithm)
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(4*1024**2), b''):
            h.update(block)
    return h.hexdigest()


def atomic_json(path, data):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix('.tmp.json')
    with open(tmp, 'w') as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
        f.flush(); os.fsync(f.fileno())
    os.replace(tmp, path)


def duration(info):
    return float(info['format']['duration'])


def validate(path, expected=None):
    info = probe(path)
    if duration(info) <= 0:
        raise ValueError('Archivo vacío')
    if expected is not None and abs(duration(info)-expected) > .25:
        raise ValueError(f'Duración inesperada: {duration(info)} vs {expected}')
    run(['ffmpeg', '-v', 'error', '-xerror', '-i', path, '-f', 'null', '-'])
    return info


def video_shape(info):
    v = next(s for s in info['streams'] if s['codec_type']=='video')
    w, h = v['width'], v['height']
    rotations = [s.get('rotation', 0) for s in v.get('side_data_list', [])]
    if any(abs(int(r))%180 == 90 for r in rotations):
        w, h = h, w
    return w, h, v.get('avg_frame_rate', v.get('r_frame_rate', '30/1'))


def normalize(source, target, shape, silence=False):
    """Keep the complete source duration; fill missing audio/video tails."""
    target = Path(target)
    info = probe(source); seconds = duration(info)
    if target.exists():
        validate(target, seconds)
        return probe(target)
    w, h, fps = shape
    partial = target.with_suffix('.partial.mp4'); partial.unlink(missing_ok=True)
    args = ['ffmpeg', '-v', 'error', '-n', '-i', source]
    audio = any(s['codec_type']=='audio' for s in info['streams']) and not silence
    if not audio:
        args += ['-f', 'lavfi', '-i', 'anullsrc=r=48000:cl=stereo']
    args += ['-map', '0:v:0', '-map', '0:a:0' if audio else '1:a:0',
             '-vf', f'scale={w}:{h}:force_original_aspect_ratio=decrease,pad={w}:{h}:(ow-iw)/2:(oh-ih)/2,setsar=1,fps={fps},tpad=stop_mode=clone:stop_duration={seconds}',
             '-af', 'aresample=48000:async=1:first_pts=0,afftdn=nr=6:nf=-35,loudnorm=I=-16:TP=-1.5:LRA=11,aresample=48000,apad',
             '-t', str(seconds), '-c:v', 'libx264', '-preset', 'veryfast', '-crf', '20',
             '-pix_fmt', 'yuv420p', '-c:a', 'aac', '-b:a', '128k', '-ar', '48000', '-ac', '2',
             '-threads', '2', '-map_metadata', '-1', '-movflags', '+faststart', partial]
    run(args)
    validate(partial, seconds); os.replace(partial, target)
    return probe(target)


def join_clips(clips, target):
    target = Path(target)
    expected = sum(duration(probe(p)) for p in clips)
    if target.exists():
        return validate(target, expected)
    partial = target.with_suffix('.partial.mp4'); partial.unlink(missing_ok=True)
    with tempfile.TemporaryDirectory() as t:
        manifest = Path(t)/'clips.txt'
        manifest.write_text(''.join("file '"+str(Path(p).resolve()).replace("'", "'\\''")+"'\n" for p in clips))
        run(['ffmpeg', '-v', 'error', '-n', '-f', 'concat', '-safe', '0', '-i', manifest,
             '-c', 'copy', '-movflags', '+faststart', partial])
    validate(partial, expected); os.replace(partial,target)
    return probe(target)


def load_asr(model_name='small', allow_openai_fallback=True):
    """CPU/int8 is reliable on standard Colab; no paid API is used."""
    from faster_whisper import WhisperModel
    try:
        model = WhisperModel(model_name, device='cpu', compute_type='int8', cpu_threads=2,
                             download_root='/content/video_studio_models')
        def infer(audio):
            segments, _ = model.transcribe(str(audio), language='es', beam_size=5,
                vad_filter=True, word_timestamps=True, condition_on_previous_text=False)
            result = []
            for s in segments:
                if s.text.strip() and not s.words:
                    raise ValueError('ASR devolvió texto sin marcas de palabra')
                result.append({'text':s.text, 'avg_logprob':s.avg_logprob,
                    'no_speech_prob':s.no_speech_prob,
                    'words':[{'word':w.word, 'start':w.start, 'end':w.end,
                              'probability':w.probability} for w in (s.words or [])]})
            return result
        return infer, 'faster-whisper:'+model_name
    except Exception as error:
        if not allow_openai_fallback:
            raise
        print('No se pudo cargar faster-whisper:', type(error).__name__)
        print('Alternativa: OpenAI Whisper local, pesos oficiales y verificación SHA-256 de su cargador.')
        run([os.sys.executable, '-m', 'pip', 'install', '-q', 'openai-whisper==20250625'])
        import whisper
        model = whisper.load_model(model_name, device='cpu', download_root='/content/video_studio_models')
        def infer(audio):
            data = model.transcribe(str(audio), language='es', fp16=False, verbose=False,
                                    word_timestamps=True, condition_on_previous_text=False)
            return data['segments']
        return infer, 'openai-whisper:'+model_name


def transcribe_clip(source, source_key, checkpoint_root, infer, engine, chunk=300):
    """2 s context around each owned interval; assign words by midpoint.
    Saves completed intervals to mounted Drive, without dropping quiet speech by hand.
    """
    checkpoint_root = Path(checkpoint_root); checkpoint_root.mkdir(parents=True, exist_ok=True)
    seconds = duration(probe(source)); words = []; flags = []
    for i in range(math.ceil(seconds/chunk)):
        owned_start, owned_end = i*chunk, min((i+1)*chunk, seconds)
        start, end = max(0, owned_start-2), min(seconds, owned_end+2)
        key = {'version':2, 'source':source_key, 'engine':engine, 'interval':[owned_start, owned_end]}
        record = checkpoint_root/f'{i:04d}.json'
        if record.exists():
            data = json.loads(record.read_text())
            if data['key'] != key:
                raise ValueError('Checkpoint incompatible. Usar otra carpeta de trabajo.')
        else:
            with tempfile.TemporaryDirectory() as t:
                audio = Path(t)/'audio.wav'
                run(['ffmpeg', '-v', 'error', '-n', '-ss', str(start), '-i', source,
                     '-t', str(end-start), '-vn', '-ac', '1', '-ar', '16000', audio])
                results = infer(audio)
            chosen = []; warnings = []
            for s in results:
                if s.get('avg_logprob', 0)<-.8 or s.get('no_speech_prob', 0)>.5:
                    warnings.append({'start':start+s['words'][0]['start'] if s.get('words') else start,
                                     'text':s.get('text',''), 'reason':'revisar_confianza'})
                for word in s.get('words', []):
                    a, b = start+float(word['start']), start+float(word['end'])
                    if owned_start <= (a+b)/2 < owned_end:
                        chosen.append({'word':word['word'], 'start':max(0,a), 'end':min(seconds,b),
                                       'probability':word.get('probability')})
            data = {'key':key, 'words':chosen, 'review':warnings}
            atomic_json(record, data)
        words += data['words']; flags += data['review']
        print(f'{Path(source).name}: intervalo {i+1}/{math.ceil(seconds/chunk)} guardado')
    return words, flags


def cues_from_words(words, width=1920, height=1080):
    line_limit = max(18, min(42, round(42*(width/height)/(16/9))))
    groups = []; current = []
    def text(group):
        return re.sub(r'\s+([,.;:!?])', r'\1', ' '.join(w['word'].strip() for w in group)).strip()
    for word in words:
        if not word['word'].strip():
            continue
        if current and (len(text(current+[word]))>line_limit*2 or
                        word['end']-current[0]['start']>5 or
                        word['start']-current[-1]['end']>.7 or
                        (text(current).endswith(('.', '?', '!', ';')) and
                         current[-1]['end']-current[0]['start']>1.2)):
            groups.append(current); current = []
        current.append(word)
    if current:
        groups.append(current)
    cues = []
    for i, group in enumerate(groups):
        start, end = max(0,group[0]['start']), max(w['end'] for w in group)
        lines = textwrap.wrap(text(group), line_limit, break_long_words=False, break_on_hyphens=False)
        if len(lines)>2:
            raise ValueError('Subtítulo excede dos líneas; revisar palabras antes de exportar')
        if i:
            start = max(start, cues[-1]['end'])
        end = max(start+.05,end)
        cues.append({'start':start,'end':end,'text':'\n'.join(lines)})
    return cues


def write_subtitles(cues, srt, ass, width, height):
    import pysubs2
    subs = pysubs2.SSAFile()
    for cue in cues:
        subs.append(pysubs2.SSAEvent(start=round(cue['start']*1000), end=round(cue['end']*1000),
                                   text=cue['text'].replace('\n',r'\N')))
    subs.info.update(PlayResX=str(width), PlayResY=str(height), WrapStyle='0')
    subs.styles['Default'] = pysubs2.SSAStyle(fontname='DejaVu Sans', fontsize=max(14,48*height/1080),
        primarycolor=pysubs2.Color(255,255,255), outlinecolor=pysubs2.Color(0,0,0),
        outline=max(1.2,2.5*height/1080), shadow=0, alignment=pysubs2.Alignment.BOTTOM_CENTER,
        marginl=round(width*.07), marginr=round(width*.07), marginv=round(height*.06))
    subs.save(str(srt)); subs.save(str(ass))


def export_burned(joined, ass, final):
    final = Path(final)
    if final.exists():
        return validate(final, duration(probe(joined)))
    partial = final.with_suffix('.partial.mp4'); partial.unlink(missing_ok=True)
    with tempfile.TemporaryDirectory() as t:
        safe = Path(t)/'captions.ass'; shutil.copy2(ass,safe)
        run(['ffmpeg','-v','error','-n','-i',joined,'-vf',f'ass={safe}',
             '-c:v','libx264','-preset','veryfast','-crf','20','-pix_fmt','yuv420p',
             '-c:a','copy','-threads','2','-map_metadata','-1','-movflags','+faststart',partial])
    validate(partial, duration(probe(joined))); os.replace(partial,final)
    return probe(final)


def frame(path, at, target):
    run(['ffmpeg','-v','error','-y','-ss',str(at),'-i',path,'-frames:v','1',target])


def pipeline(sources, local_root, durable_root, infer, engine, intro=None):
    """sources are already reviewed/ordered; all originals remain untouched."""
    local_root = Path(local_root); durable_root = Path(durable_root)
    local_root.mkdir(parents=True,exist_ok=True); durable_root.mkdir(parents=True,exist_ok=True)
    source_hashes = [checksum(p) for p in sources]
    details = [probe(p) for p in sources]
    if len(sources)!=3 or len(set(source_hashes))!=3:
        raise ValueError('Se requieren tres archivos distintos')
    shape = video_shape(details[0])
    shape = (shape[0]+shape[0]%2, shape[1]+shape[1]%2, shape[2])
    if not Fraction(shape[2]):
        shape = (shape[0],shape[1],'30/1')
    intro_hash = checksum(intro) if intro else None
    key = {'sources':source_hashes,'shape':list(shape),'intro':intro_hash,'engine':engine,'version':2}
    lockfile = durable_root/'job.json'
    if lockfile.exists() and json.loads(lockfile.read_text())!=key:
        raise ValueError('Otro orden/configuración ya usa esta carpeta. Elegir una nueva carpeta de trabajo.')
    atomic_json(lockfile,key)
    required = sum(Path(p).stat().st_size for p in sources)*4 + 1024**3
    if shutil.disk_usage(local_root).free < required:
        raise RuntimeError(f'Espacio local insuficiente. Estimación mínima: {required/1024**3:.1f} GiB')
    if shutil.disk_usage(durable_root).free < required:
        raise RuntimeError('Espacio insuficiente en Drive para checkpoints y entrega')
    normalized = []; rows = []; all_words = []; review = []; offset = 0
    if intro:
        p = local_root/'intro.mp4'; d = normalize(intro,p,shape,silence=True)
        normalized.append(p); offset = duration(d)
    for i, (source, info, sha) in enumerate(zip(sources,details,source_hashes)):
        target = local_root/f'normalized_{i}.mp4'
        d = normalize(source,target,shape)
        normalized.append(target)
        rows.append({'source':Path(source).name,'sha256':sha,'source_duration':duration(info),
                     'output_start':offset,'output_end':offset+duration(d)})
        words, warnings = transcribe_clip(target,sha,durable_root/f'checkpoints_{i}',infer,engine)
        all_words += [{**w,'start':w['start']+offset,'end':w['end']+offset} for w in words]
        review += [{**w,'start':w['start']+offset} for w in warnings]
        offset += duration(d)
    joined = local_root/'joined.mp4'; join_clips(normalized,joined)
    cues = cues_from_words(all_words,*shape[:2])
    if not cues:
        raise ValueError('No hay subtítulos reconocidos; no se exporta una entrevista muda de texto')
    if cues[-1]['end']>duration(probe(joined))+.1:
        raise ValueError('Subtítulos fuera de la duración')
    srt = local_root/'ENTREVISTA_MALVINAS_SUBTITULOS.srt'
    ass = local_root/'captions.ass'; write_subtitles(cues,srt,ass,*shape[:2])
    final = local_root/'ENTREVISTA_MALVINAS_FINAL.mp4'; final_info = export_burned(joined,ass,final)
    streams = final_info['streams']
    if next(s for s in streams if s['codec_type']=='video')['codec_name']!='h264':
        raise ValueError('Codec de video incorrecto')
    if next(s for s in streams if s['codec_type']=='audio')['codec_name']!='aac':
        raise ValueError('Codec de audio incorrecto')
    for i, cue in enumerate(cues):
        cps = len(cue['text'].replace('\n',''))/(cue['end']-cue['start'])
        if cps>22:
            review.append({'cue':i+1,'start':cue['start'],'reason':'velocidad_lectura','cps':round(cps,1)})
    report = {'clips':rows,'intro_seconds':rows[0]['output_start'],
              'expected_seconds':sum(duration(probe(p)) for p in normalized),
              'final_seconds':duration(final_info),'decode_complete':True,
              'subtitles_count':len(cues),'human_sync_review':False,'review':review,
              'final_sha256':checksum(final),'srt_sha256':checksum(srt)}
    # Difference test and saved frames at spread-out, actually captioned times.
    import numpy as np
    shots = []; seen = set()
    for index in [0,len(cues)//4,len(cues)//2,3*len(cues)//4,len(cues)-1]:
        cue = cues[index]; at = (cue['start']+cue['end'])/2
        if index in seen: continue
        seen.add(index)
        def pixels(path):
            return subprocess.run(['ffmpeg','-v','error','-ss',str(at),'-i',str(path),
               '-frames:v','1','-f','rawvideo','-pix_fmt','gray','-'],check=True,capture_output=True).stdout
        before = np.frombuffer(pixels(joined),dtype=np.uint8).astype(np.int16)
        after = np.frombuffer(pixels(final),dtype=np.uint8).astype(np.int16)
        changed = int(np.count_nonzero(np.abs(after-before)>35))
        if changed < 10:
            raise ValueError('No se detectó texto quemado en el fotograma esperado')
        shot = local_root/f'check_{index:04d}.png'; frame(final,at,shot)
        shots.append({'at':at,'file':str(shot),'cue':cue['text'],'changed_pixels':changed})
    report['subtitle_frame_checks']=shots
    atomic_json(durable_root/'validation.json',report)
    # Save recoverable text immediately, before uploading the large final file.
    shutil.copy2(srt,durable_root/srt.name)
    atomic_json(durable_root/'words.json',all_words)
    return final,srt,report


In [ ]:
# 2 — Drive autenticado. No pegues tokens, contraseñas ni cookies.
from google.colab import drive, auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload, MediaFileUpload
import google.auth
from datetime import datetime
import time

drive.mount('/content/drive')
auth.authenticate_user()
credentials,_=google.auth.default()
service=build('drive','v3',credentials=credentials,cache_discovery=False)

FOLDER_URL=input('Pegá la URL de la carpeta privada de Drive con los tres videos: ').strip()
match=re.search(r'/folders/([A-Za-z0-9_-]+)',FOLDER_URL)
assert match,'La URL debe identificar una carpeta de Drive.'
FOLDER_ID=match.group(1)
EXPECTED=['VID-20261107-WA0034.mp4','VID-20261107-WA0035.mp4','VID-20261107-WA0036.mp4']
# Para retomar tras desconexión, copiá aquí el nombre de carpeta que se imprime abajo.
RESUME_FOLDER_NAME=''
LOCAL=Path('/content/video_studio_malvinas'); LOCAL.mkdir(exist_ok=True)
INPUT=LOCAL/'input'; INPUT.mkdir(exist_ok=True)

if RESUME_FOLDER_NAME:
    safe=RESUME_FOLDER_NAME.replace("'","\'")
    matches=service.files().list(q=f"name = '{safe}' and 'root' in parents and trashed = false",fields='files(id,name)',pageSize=100).execute()['files']
    assert len(matches)==1,'No hay una carpeta única con ese nombre en Mi unidad.'
    remote_folder=matches[0]
else:
    name='VideoStudioAI_Malvinas_'+datetime.now().strftime('%Y%m%d_%H%M%S')
    remote_folder=service.files().create(body={'name':name,'mimeType':'application/vnd.google-apps.folder'},fields='id,name').execute()

permissions=service.permissions().list(fileId=remote_folder['id'],fields='permissions(type,role)').execute()['permissions']
assert permissions and all(p['type']=='user' and p['role']=='owner' for p in permissions),'La carpeta de salida debe ser privada, solo de su propietario.'
DURABLE=Path('/content/drive/MyDrive')/remote_folder['name']
for attempt in range(15):
    if DURABLE.exists():break
    time.sleep(1)
assert DURABLE.exists(),'Drive todavía no mostró la carpeta montada; reejecutá esta celda con RESUME_FOLDER_NAME.'
print('CARPETA PARA REANUDAR:',remote_folder['name'])
print('Salida privada:', 'https://drive.google.com/drive/folders/'+remote_folder['id'])

# Acceso real con tu cuenta; ni la carpeta ni los videos se cambian a públicos.
items=[]; token=None
while True:
    result=service.files().list(q=f"'{FOLDER_ID}' in parents and trashed=false",pageToken=token,
        pageSize=1000,includeItemsFromAllDrives=True,supportsAllDrives=True,
        fields='nextPageToken,files(id,name,size,md5Checksum,mimeType,capabilities/canDownload)').execute()
    items+=result.get('files',[]);token=result.get('nextPageToken')
    if not token:break
selected=[]
for name in EXPECTED:
    matches=[f for f in items if f['name']==name]
    assert len(matches)==1,f'Falta el archivo o hay duplicados: {name}'
    info=matches[0]
    assert info.get('capabilities',{}).get('canDownload',True),f'La cuenta no tiene permiso de descarga: {name}'
    assert int(info.get('size',0))>0,f'Archivo vacío/no descargable: {name}'
    selected.append(info)
required=sum(int(f['size']) for f in selected)*5+1024**3
assert shutil.disk_usage(LOCAL).free>required,'Se necesita más espacio local para esta entrevista.'
originals=[]
for info in selected:
    path=INPUT/info['name']
    valid=path.exists() and path.stat().st_size==int(info['size'])
    if valid and info.get('md5Checksum'):valid=checksum(path,'md5')==info['md5Checksum']
    if not valid:
        partial=path.with_suffix('.partial'); partial.unlink(missing_ok=True)
        with open(partial,'wb') as f:
            downloader=MediaIoBaseDownload(f,service.files().get_media(fileId=info['id'],supportsAllDrives=True),chunksize=16*1024**2)
            done=False
            while not done:
                progress,done=downloader.next_chunk(num_retries=3)
                print(info['name'],round(progress.progress()*100),'%')
        assert partial.stat().st_size==int(info['size']),'Tamaño descargado incorrecto'
        if info.get('md5Checksum'):assert checksum(partial,'md5')==info['md5Checksum'],'Checksum de origen incorrecto'
        validate(partial)
        os.replace(partial,path)
    originals.append(path)
atomic_json(DURABLE/'source_manifest.json',selected)
print('Tres copias descargadas, tamaño/integridad verificados. Originales remotos intactos.')


In [ ]:
# 3 — Inspección de continuidad. No se presupone el orden de los nombres.
from IPython.display import display, Video, Markdown, Image
for number,path in enumerate(originals,1):
    info=probe(path); secs=duration(info)
    display(Markdown(f'### {number}: {path.name} — {secs:.2f} s — {video_shape(info)}'))
    print('Creación declarada:',info['format'].get('tags',{}).get('creation_time','sin dato fiable'))
    for label,start in [('principio',0),('final',max(0,secs-15))]:
        preview=LOCAL/f'preview_{number}_{label}.mp4'
        if not preview.exists():
            run(['ffmpeg','-v','error','-n','-ss',str(start),'-i',path,'-t','15',
                 '-vf','scale=640:-2','-c:v','libx264','-preset','veryfast','-crf','26',
                 '-c:a','aac','-b:a','96k','-threads','2',preview])
        print(label)
        display(Video(str(preview),embed=True,width=640))
print('Escuchá la continuidad de las intervenciones antes de elegir. No se recorta ninguna declaración.')
raw=input('Orden confirmado por el contenido (números 1,2,3 en el orden correcto, separados por comas): ')
order=[int(x.strip())-1 for x in raw.split(',')]
assert sorted(order)==[0,1,2],'El orden debe incluir exactamente una vez cada archivo.'
sources=[originals[i] for i in order]
atomic_json(DURABLE/'order_review.json',{'order':[p.name for p in sources],'reviewed_by_user':True})
print('Orden elegido:',[p.name for p in sources])


In [ ]:
# Apertura opcional existente. No hay videos embebidos en este notebook.
# Si subiste la apertura creada a la raíz de Mi unidad, se reutiliza; si no, se omite.
INTRO=Path('/content/drive/MyDrive/caratula_malvinas_3s.mp4')
if not INTRO.exists():
    INTRO=None
    print('Apertura no disponible en esta sesión Colab: se prioriza la entrevista completa.')


In [ ]:
# 4 — Transcripción y exportación completa. Puede tardar horas en CPU.
# Modelo small multilingüe, CPU/int8; no se usa small.en ni servicios pagos.
infer,engine=load_asr('small',allow_openai_fallback=True)
print('Motor:',engine)
FINAL,SRT,REPORT=pipeline(sources,LOCAL/'render',DURABLE/'trabajo',infer,engine,intro=INTRO)
print('MP4 creado y decodificado completamente:',FINAL)
print('Duración:',REPORT['final_seconds'],'s; subtítulos:',REPORT['subtitles_count'])
print('Segmentos incluidos:',[r['source'] for r in REPORT['clips']])
print('Revisiones sugeridas:',len(REPORT['review']))
display(REPORT['review'][:30])


In [ ]:
# 5 — Comprobación visual y de sincronización con voz real.
for shot in REPORT['subtitle_frame_checks']:
    print(f"{shot['at']:.2f}s — {shot['cue']}")
    display(Image(filename=shot['file']))
# Ventanas subtituladas y uniones entre grabaciones, con audio, no la hora embebida.
review_times=[s['at'] for s in REPORT['subtitle_frame_checks']]
review_times += [r['output_start'] for r in REPORT['clips'][1:]]
for i,at in enumerate(review_times):
    sample=LOCAL/f'revision_sincronia_{i}.mp4'
    if not sample.exists():
        run(['ffmpeg','-v','error','-n','-ss',str(max(0,at-3)),'-i',FINAL,'-t','8',
             '-vf','scale=640:-2','-c:v','libx264','-preset','veryfast','-crf','25',
             '-c:a','aac','-threads','2',sample])
    display(Video(str(sample),embed=True,width=640))
print('Revisá voz natural, continuidad, correspondencia del texto y nombres/fechas/lugares en el SRT.')
print('SRT editable guardado en:',DURABLE/'trabajo'/SRT.name)
print('Para corregirlo: editar ese SRT en Drive y ejecutar la celda siguiente.')


In [ ]:
# Opcional — aplicar correcciones humanas del SRT; no retranscribe.
APLICAR_SRT_CORREGIDO=False
if APLICAR_SRT_CORREGIDO:
    import pysubs2
    reviewed=pysubs2.load(str(DURABLE/'trabajo'/SRT.name))
    cues=[{'start':s.start/1000,'end':s.end/1000,'text':s.plaintext} for s in reviewed]
    previous=0
    for c in cues:
        assert c['start']>=previous and c['end']>c['start'] and c['end']<=REPORT['final_seconds']+.1,'Corregir tiempos del SRT.'
        assert len(c['text'].splitlines())<=2,'Usar como máximo dos líneas.'
        previous=c['end']
    w,h,_=video_shape(probe(FINAL))
    corrected=LOCAL/'render'/'captions_corrected.ass'
    write_subtitles(cues,SRT,corrected,w,h)
    updated=LOCAL/'render'/'ENTREVISTA_MALVINAS_REVISADA.mp4'
    if updated.exists():updated.unlink() # solo derivado local de esta sesión
    export_burned(LOCAL/'render'/'joined.mp4',corrected,updated)
    os.replace(updated,FINAL)
    REPORT['final_sha256']=checksum(FINAL);REPORT['srt_sha256']=checksum(SRT)
    print('Correcciones aplicadas. Volvé a ejecutar la celda 5 para inspeccionarlas.')


In [ ]:
# 6 — Entrega real: subida autenticada, privada, con comprobación del servidor.
answer=input('¿Revisaste continuidad, voz y sincronización del texto en las muestras? Escribí SI para entregar: ')
assert answer.strip().upper()=='SI','La validación humana queda pendiente.'
REPORT['human_sync_review']=True
atomic_json(DURABLE/'trabajo'/'validation.json',REPORT)
links=[]
for path in [FINAL,SRT]:
    media=MediaFileUpload(str(path),mimetype='video/mp4' if path.suffix=='.mp4' else 'application/x-subrip',chunksize=8*1024**2,resumable=True)
    matches=service.files().list(q=f"'{remote_folder['id']}' in parents and name='{path.name}' and trashed=false",
        fields='files(id,size,md5Checksum)',pageSize=100).execute()['files']
    assert len(matches)<=1,'Hay duplicados de este entregable; usá una carpeta nueva para evitar ambigüedad.'
    result=None
    if matches and matches[0].get('md5Checksum')==checksum(path,'md5') and int(matches[0]['size'])==path.stat().st_size:
        result=matches[0]
    else:
        if matches:
            request=service.files().update(fileId=matches[0]['id'],media_body=media,fields='id,name,size,md5Checksum,webViewLink')
        else:
            request=service.files().create(body={'name':path.name,'parents':[remote_folder['id']]},media_body=media,
                                           fields='id,name,size,md5Checksum,webViewLink')
        while result is None:
            status,result=request.next_chunk(num_retries=3)
            if status:print(path.name,round(status.progress()*100),'% cargado')
    stored=service.files().get(fileId=result['id'],fields='id,name,size,md5Checksum,webViewLink').execute()
    assert int(stored['size'])==path.stat().st_size,'Drive no confirmó el tamaño correcto.'
    assert stored['md5Checksum']==checksum(path,'md5'),'Drive no confirmó el checksum correcto.'
    perms=service.permissions().list(fileId=stored['id'],fields='permissions(type,role)').execute()['permissions']
    assert all(p['type']=='user' and p['role']=='owner' for p in perms),'Revisar privacidad de la entrega.'
    links.append(stored)
    print('CARGA VERIFICADA:',stored['name'],stored['webViewLink'])
atomic_json(DURABLE/'entrega_verificada.json',{'files':links,'validation':REPORT})
print('Los tres fragmentos están incluidos; MP4/SRT cargados y verificados en Drive privado.')
print('Abrí los enlaces con la misma cuenta desde Drive en tu celular.')
print('Android: menú ⋮ del MP4 → Descargar. iPhone: menú ⋮ → Enviar una copia/Abrir en → Guardar en Archivos o Guardar video.')
print('La descarga en el dispositivo se confirma desde tu celular; el notebook confirma la carga y el checksum en Drive.')
